# 6.5.8 Embodied AI and Robotics

Grid-world navigation with REINFORCE (policy gradient). Train a softmax policy on a 6×6 grid and visualise the learned action map.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]; NAMES = ['↑','↓','←','→']
SIZE = 6; GOAL = (5,5); OBS = {(2,2),(2,3),(3,2),(4,4)}

def step(pos, action):
    dr,dc = ACTIONS[action]; nr,nc = pos[0]+dr, pos[1]+dc
    if 0<=nr<SIZE and 0<=nc<SIZE and (nr,nc) not in OBS: pos = (nr,nc)
    done = pos==GOAL; return pos, 10. if done else -0.1, done

def sfmx(x): e=np.exp(x-x.max()); return e/e.sum()

theta = rng.standard_normal((SIZE*SIZE, 4))*0.1
rewards_all = []
for ep in range(200):
    pos=(0,0); lps,rws=[],[]
    for _ in range(80):
        si=pos[0]*SIZE+pos[1]; p=sfmx(theta[si]); a=rng.choice(4,p=p)
        lps.append(np.log(p[a]+1e-10)); pos,r,done=step(pos,a); rws.append(r)
        if done: break
    rewards_all.append(sum(rws))
    G=0; ret=[]
    for r in reversed(rws): G=r+0.95*G; ret.insert(0,G)
    bl = np.mean(ret)
    for lp,g in zip(lps,ret): theta+=0.02*(g-bl)
print(f'Ep 200 total reward: {rewards_all[-1]:.2f}')

In [ ]:
def ma(x,w=20): return np.convolve(x,np.ones(w)/w,'valid')
policy = np.argmax([sfmx(theta[s]) for s in range(SIZE*SIZE)],1).reshape(SIZE,SIZE)

fig, axes = plt.subplots(1,2,figsize=(12,5))
axes[0].plot(rewards_all, alpha=0.3, color='steelblue')
axes[0].plot(range(19,200), ma(rewards_all), color='steelblue', lw=2)
axes[0].set(xlabel='Episode', ylabel='Reward', title='Policy Gradient Training')
axes[0].grid(True, alpha=0.3)

gd = np.full((SIZE,SIZE),0.5)
for r,c in OBS: gd[r,c]=0.0
gd[GOAL]=1.0
axes[1].imshow(gd, cmap='RdYlGn', vmin=0, vmax=1)
for r in range(SIZE):
    for c in range(SIZE):
        if (r,c) in OBS: axes[1].text(c,r,'X',ha='center',va='center',fontsize=12,fontweight='bold')
        elif (r,c)==GOAL: axes[1].text(c,r,'G',ha='center',va='center',fontsize=12,color='white',fontweight='bold')
        else: axes[1].text(c,r,NAMES[policy[r,c]],ha='center',va='center',fontsize=14)
axes[1].set(title='Learned Policy', xticks=[], yticks=[])
plt.tight_layout()
plt.savefig('output/embodied_ai.png')
print('Saved embodied_ai.png')